# 다중시점 historical CLV 조건부 LightGCN M2 — Dunnhumby seed 42

한 사용자의 동일한 64차원 ID 임베딩을 여러 과거 시점에서 공유하되, 그 시점까지 누적된 historical CLV proxy에 따라 임베딩의 차원별 사용 비율을 조금씩 변화시키는 M2 탐색 실험입니다.

- 시점별 historical CLV proxy: $C_{u,t}=N_{u,t}\times V_{u,t}$
  - $N_{u,t}$: 기준시점까지 누적 장바구니 횟수의 백분위
  - $V_{u,t}$: 기준시점까지 평균 장바구니 구매금액의 백분위
- 사용자 layer-0: $\widetilde E_{u,t}=\operatorname{NormPreserve}[E_u^{ID}\odot(1+\rho c_{u,t}\tanh(w))]$
- $c_{u,t}=2\operatorname{Percentile}(C_{u,t})-1$, $\rho=0.05$ 고정
- 아이템은 순수 ID 임베딩만 사용; 별도 CLV 점수나 아이템 가격 변수 없음
- 학습 시점: 655·662·669·676일 이전 이력 → 각각 다음 7일의 신규상품 쌍
- 최종 탐색 평가: 1~683일 이력, 684~690일 신규상품
- 대조군: 같은 다중시점 그래프·정답·학습순서에서 $\rho=0$
- 고정: binary graph, uniform negative sampling, plain BPR, 표본 가중 없음, 100 epoch, 하나의 optimizer

이는 역사적 개발구간 seed 42 탐색이며, 최종 test를 만들지 않고 유의성·일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'b399abf4cd7bed8069353e948c47779e936c1cfa'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_dynamic_multianchor import (
    configure_dynamic_multianchor,
    preflight_summary,
    run_dynamic_multianchor,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_dynamic_multianchor(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_dynamic_clv_multianchor_historical_screen_v1'
    )
)
summary = preflight_summary(cfg)
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m2']['rho'] == 0.05
assert summary['m2']['item_side_CLV_input'] is False
assert summary['m2']['separate_CLV_score'] is False
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['new_loss_term'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_dynamic_multianchor(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
decision = dict(result_df.attrs['decision'])
paths = dict(result_df.attrs['result_files'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))
core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('동일 다중시점 protocol의 rho=0 대조군 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', decision)
print('결과 파일:', paths)